# RQ1: transfer — results

Five-fold evaluation of the reference recipe (E1) against the original models on identical context and test rows: per-table effects and their seed noise (E2), transfer against forgetting, discrimination against calibration and distribution quality, the position relative to tuned classical baselines, the matched non-credit control (E4) and fine-tuning on the target table (E5).

Unit of analysis: one value per held-out table (folds are averaged first; only tables with all five paired folds enter). PD reports AUC, LGD R².

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pandas as pd
from src.visualize import style, campaign as cp, campaign2 as c2, results2 as r2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('rq1/02_results')
report = cp.NotebookReport('RQ1: transfer — results', sink=sink)
gains = {track: c2.endpoint_gains('reference', track) for track in ('pd', 'lgd')}
seeds = {track: c2.seed_noise(track) for track in ('pd', 'lgd')}

## 1. Per-table transfer and seed noise (E1, E2)

Question: on which held-out credit tables does CPT change the five-fold score, by how much, and is the change larger than retraining with another seed moves it?

In [2]:
for track in ('pd', 'lgd'):
    fig = r2.table_gains(gains[track], seeds[track], track=track,
                         title='Effect of CPT per held-out credit table (E1, E2)')
    cp.show(sink, c2.pages(fig, 'reference_tables',
        'Five-fold change of ' + r2.SCORE[track] + ' from continued pretraining with the reference recipe, per held-out '
        'credit table (rows) and base model (columns): CPT minus the original model on identical context and test rows, '
        'averaged over the five outer folds (seed 42, E1). Horizontal lines span the changes of seeds 42, 43 and 44 '
        '(E1, E2); a filled marker means all three seeds agree in sign and the smallest absolute change exceeds their '
        'range. The dashed line is the equal-table mean of seed 42. Only tables with all five paired folds are shown; '
        'horizontal scales differ between panels.'), track=track)
    report.table(track.upper() + ' E1 summary per model (credit and retention tables)', c2.gain_summary(gains[track]))
    report.table(track.upper() + ' E1 per-table gains', gains[track])
    report.table(track.upper() + ' E2 summary per seed arm', c2.gain_summary(c2.endpoint_gains('seed', track)))
    report.table(track.upper() + ' gains at three seeds', seeds[track])
intervals = {track: c2.gain_intervals('reference', track) for track in ('pd', 'lgd')}
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' mean gain with 95% bootstrap interval (E1)', intervals[track])
report.add('1. Per-table transfer and seed noise (E1, E2)', c2.evidence_note('E1 five-fold gains', gains['pd']))

No measurements available for this section yet.


""


""


""


""


No measurements available for this section yet.


... (2 outputs left out)

""


""


""


""


## 2. Transfer against forgetting, with uncertainty

Question: how large is the mean effect of CPT on held-out credit tables and on the non-credit retention tables, how uncertain is it given the number of tables, and how do individual tables spread around it?

In [3]:
fig = r2.transfer_and_retention(gains, intervals, title='Effect of CPT on credit and on non-credit tables (E1)')
cp.show(sink, c2.pages(fig, 'transfer_and_retention',
    'Five-fold change from continued pretraining with the reference recipe (E1, seed 42), CPT minus the original model '
    'on identical rows, per base model: held-out credit tables (filled, left; 17 PD, 8 LGD) and non-credit TabArena '
    'retention tables (hollow, right; 19 binary, 12 regression; averaged over the four partition models). Diamonds are '
    'equal-table means; vertical bars are their 95% percentile intervals from a hierarchical bootstrap (2,000 draws) '
    'that resamples tables and, within each drawn table, its five folds. PD reports AUC, LGD R²; only tables with all '
    'five paired folds are shown.'))
report.add('2. Transfer against forgetting, with uncertainty', c2.evidence_note('E1 retention gains', c2.select(gains['pd'], credit=False)))

No measurements available for this section yet.


## 3. Discrimination, calibration and distribution quality

Question: does CPT change probability calibration (PD) or the predictive distribution (LGD) differently from discrimination or point accuracy? Each metric is oriented so that a positive value is an improvement.

In [4]:
metrics = {track: c2.endpoint_metrics('reference', track) for track in ('pd', 'lgd')}
profiles = {track: c2.select(c2.metric_profile(metrics[track]), arm=0) for track in ('pd', 'lgd')}
fig = r2.metric_profile_heatmap(profiles, title='Which aspects of prediction quality CPT changes (E1, held-out credit tables)')
cp.show(sink, c2.pages(fig, 'metric_profile',
    'Five-fold effect of continued pretraining with the reference recipe (E1) on discrimination and on probability or '
    'distribution quality, per base model (rows) and metric (columns), over the held-out credit tables with all five '
    'paired folds. Colour: share of tables on which CPT improved the metric; text: equal-table mean improvement and '
    'the number of improved tables. Improvements are oriented so that positive is better: increase in AUC and R²; '
    'decrease in log loss, Brier score, ECE, normalised RMSE, MAE and CRPS; decrease in |calibration slope − 1|, '
    '|calibration-in-the-large|, |mean bias| and |prediction SD / target SD − 1|.'))
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' metric profile (credit tables)', profiles[track])
    report.table(track.upper() + ' metric profile (retention tables)', c2.select(c2.metric_profile(metrics[track], credit=False), arm=0))
report.add('3. Discrimination, calibration and distribution quality', c2.evidence_note('E1 metric profile', profiles['pd']))

No measurements available for this section yet.


""


""


""


""


## 4. Position relative to tuned classical baselines

Question: where do the original and the CPT models stand relative to tuned CatBoost and XGBoost (50 Optuna trials each) on the same folds, and does CPT change that position? The linear baseline (logistic or linear regression) is reported in the table.

In [5]:
baselines = {track: c2.baseline_comparison(track) for track in ('pd', 'lgd')}
fig = r2.baseline_position(baselines, title='Original and CPT models relative to tuned CatBoost (E1)')
cp.show(sink, c2.pages(fig, 'baselines',
    'Five-fold score of the original (hollow) and the CPT model (filled, E1 reference) minus the score of tuned CatBoost '
    'on the same folds, per held-out credit table (grey lines join one table), task (rows; AUC for PD, R² for LGD) and '
    'base model (columns). Diamonds are equal-table means; zero is CatBoost. Only tables on which all four models were '
    'scored on all five folds are shown.'))
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' scores relative to CatBoost per table (with the XGBoost and linear baselines)', baselines[track])
report.add('4. Position relative to tuned classical baselines', c2.evidence_note('baseline comparison', baselines['pd']))

No measurements available for this section yet.


""


""


## 5. Credit versus matched non-credit CPT (E4)

Question: is the transfer domain-specific? Each held-out credit table is scored after CPT on the credit training tables (E1) and after CPT on their matched non-credit twins (E4), on the same folds and rows.

In [6]:
controls = {track: c2.domain_control_gains(track) for track in ('pd', 'lgd')}
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' E4 versus E1 per table', controls[track])
    report.table(track.upper() + ' E4 versus E1 per model', controls[track].groupby('model')[['gain_credit', 'gain_noncredit', 'credit_minus_noncredit']].agg(['mean', 'median', 'size']).reset_index() if not controls[track].empty else controls[track])
control_intervals = {track: c2.domain_control_intervals(track) for track in ('pd', 'lgd')}
fig = r2.gain_forest(control_intervals, value='credit_minus_noncredit',
                     xlabel='Credit CPT gain − non-credit CPT gain',
                     title='Is the effect specific to credit data? (E1 minus E4)')
cp.show(sink, c2.pages(fig, 'domain_control_difference',
    'Difference in differences per base model on the held-out credit tables: the five-fold gain after CPT on '
    'the credit training tables (E1) minus the gain after CPT on their matched non-credit twins (E4), computed '
    'fold by fold on identical rows and averaged with equal table weight. Lines are 95% percentile intervals of '
    'a hierarchical bootstrap over tables and folds (2,000 draws). Positive values mean that credit data helped '
    'more than comparable non-credit data. AUC for PD, R² for LGD.'))
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' credit minus non-credit CPT gain with 95% bootstrap interval', control_intervals[track])
report.add('5. Credit versus matched non-credit CPT (E4)', c2.evidence_note('E4 domain control', controls['pd']))

""


""


""


""


No measurements available for this section yet.


""


""


## 6. CPT versus fine-tuning on the target table (E5)

Question: is CPT worth it compared with fine-tuning on the target table's own training rows, and do the two stack? All four conditions share the context cap, folds and test rows.

In [7]:
results = c2.task_results('finetune')
finetune = c2.finetune_table(results)
fig = r2.finetune_conditions(finetune, title='CPT, target fine-tuning and both (E5)')
cp.show(sink, c2.pages(fig, 'target_finetune',
    'Five-fold change relative to the original model without fine-tuning, per held-out credit table (grey lines join '
    'one table), task (rows; AUC for PD, R² for LGD) and base model (columns): CPT alone (E1), the original model '
    'fine-tuned on the fold\'s training rows (FT; learning rate chosen from 1e-6, 1e-5 and 6.5e-5 on the fold\'s validation rows, up to 30 epochs '
    'with early stopping) and the CPT model fine-tuned the same way (CPT+FT). All conditions use the same context cap, '
    'validation and test rows; diamonds are equal-table means over tables with all five folds.'))
report.table('E5 summary per model and start', c2.finetune_summary(results))
report.table('E5 per table', finetune)
cost = c2.cost_comparison(c2.run_resources(c2.load_trajectories(['reference'])), results)
report.table('GPU hours: CPT (E1, paid once) against fine-tuning per target table (E5)', cost)
report.add('6. CPT versus fine-tuning on the target table (E5)', c2.evidence_note('E5 fine-tuning', finetune))

No measurements available for this section yet.

""


""


""


## 7. Training precision (E3)

Question: does CPT under BF16 (TabPFN v2.6, v3) or under strict FP32 (TabICLv2) end somewhere else than the FP16 reference, on identical data, updates and seed? A numerical control, like the seeds of E2.

In [8]:
for track in ('pd', 'lgd'):
    precision = c2.arms_overview(['reference', 'precision'], track)
    report.table(track.upper() + ' five-fold effect per training precision and model (E1 reference: FP16; E3)', precision)
    report.table(track.upper() + ' E3 mean gain with 95% bootstrap interval', c2.gain_intervals('precision', track))
    report.table(track.upper() + ' E3 per-table gains', c2.endpoint_gains('precision', track))
report.add('7. Training precision (E3)', c2.evidence_note('E3 results', c2.endpoint_gains('precision', 'pd')))

""


""


""


""


""


""


## Summary

Every displayed table and figure, in section order.

In [9]:
print(report.summary(sink))

RQ1: transfer — results

1. Per-table transfer and seed noise (E1, E2)
No results yet for E1 five-fold gains; the figure is drawn once they exist.

No measurements available for this figure group.

Table: PD E1 summary per model (credit and retention tables)
CSV (first column: row index)
row

Table: PD E1 per-table gains
CSV (first column: row index)
row

Table: PD E2 summary per seed arm
CSV (first column: row index)
row

Table: PD gains at three seeds
CSV (first column: row index)
row

No measurements available for this figure group.

Table: LGD E1 summary per model (credit and retention tables)
CSV (first column: row index)
row

Table: LGD E1 per-table gains
CSV (first column: row index)
row

Table: LGD E2 summary per seed arm
CSV (first column: row index)
row

Table: LGD gains at three seeds
CSV (first column: row index)
row

Table: PD mean gain with 95% bootstrap interval (E1)
CSV (first column: row index)
row

Table: LGD mean gain with 95% bootstrap interval (E1)
CSV (first colum